# Session A2: bigger MIDOG++ training set + classifier crops (CPU only)

Same pipeline as Session A, with two changes:
* **20 images per tumor type** (119 train images instead of 49). The **val and test images are exactly
  the same as in Session A**, so the new results are directly comparable.
* Saves **128 x 128 crops** around every mitosis and look-alike "imposter" cell (train + val) for the
  second-stage classifier.

Optional but recommended: **Add Input** -> Your Work -> your Session A notebook. The notebook then
double-checks that val/test are identical to Session A (it stops if they are not).

**Before running**
1. Right panel, *Session options*: **Accelerator = None** (saves your GPU quota), **Internet = On**.
2. Click **Save Version** (top right), choose **Save & Run All (Commit)**. It runs in the background, so you can close the tab and sleep.
3. When it finishes, the *Output* tab has `midog_yolo.zip`, `manifest.json`, `split.csv` and `label_preview.jpg`.

Re-running is safe: images already processed are skipped.

In [ ]:
# Settings
PER_TYPE = 20          # images per tumor type (17 train / 1 val / 2 test)
BASE     = 10          # keep the val/test images of the Session A run (which used 10 per type)
CROP     = 128         # crop size for the second-stage classifier
WORKERS  = 4           # parallel downloads
WORK     = '/kaggle/working'
OUT      = f'{WORK}/midog_yolo'
TMP      = '/tmp/midog_download'

In [ ]:
# Official MIDOG++ repo: annotations (MIDOG++.json) and the figshare file list
!git clone -q --depth 1 https://github.com/DeepMicroscopy/MIDOGpp.git /tmp/MIDOGpp || echo "already cloned"
!pip install -q tifffile imagecodecs
!ls /tmp/MIDOGpp/databases

In [ ]:
%%writefile /tmp/prepare_midog.py
"""
MIDOG++ -> YOLO dataset builder (streaming, low disk usage).

What it does
  1. Picks a balanced subset of images per tumor type (or all of them).
  2. Splits by IMAGE (never by tile) into train / val / test, so no tissue
     from one image leaks into another split.
  3. For each image: download the TIFF from figshare, cut it into overlapping
     tiles, write YOLO labels, then delete the TIFF (train images) or keep it
     (val/test images, needed for full-image evaluation).
  4. Writes data.yaml for Ultralytics and a manifest with dataset statistics.

Peak disk usage is roughly (workers x one TIFF) plus the tiles, instead of 65 GB.
Re-running the script resumes: images already tiled are skipped.

Usage
  git clone https://github.com/DeepMicroscopy/MIDOGpp
  python prepare_midog.py --repo MIDOGpp --out datasets/midog_yolo --per-type 10

Notes on the MIDOG++ annotations
  * Boxes in MIDOG++.json are [x1, y1, x2, y2] (always 50 x 50 px around the
    cell), NOT the usual COCO [x, y, w, h]. They are converted accordingly.
  * Category 1 = mitotic figure (our only class), category 2 = "imposter"
    (look-alike, non-mitotic cell). Imposters are not labelled, but tiles that
    contain them are always kept as hard negatives.
"""

import argparse
import ast
import csv
import json
import os
import random
import shutil
import sys
import threading
import time
import urllib.request
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
from PIL import Image

Image.MAX_IMAGE_PIXELS = None
FIGSHARE_URL = "https://ndownloader.figshare.com/files/{}"
MITOSIS, IMPOSTER = 1, 2


# --------------------------------------------------------------------------- #
# Metadata
# --------------------------------------------------------------------------- #
def load_figshare_ids(repo: Path) -> dict:
    """Read the file_name -> figshare id map from the official Setup.ipynb."""
    nb = json.loads((repo / "Setup.ipynb").read_text())
    for cell in nb["cells"]:
        src = "".join(cell["source"])
        if src.lstrip().startswith("filename"):
            mapping = ast.literal_eval(src.split("=", 1)[1].strip())
            return {name: fid for fid, name in mapping.items()}
    sys.exit("Could not find the figshare file list in Setup.ipynb")


def load_annotations(repo: Path):
    data = json.loads((repo / "databases" / "MIDOG++.json").read_text())
    images = {im["id"]: im for im in data["images"]}
    boxes = defaultdict(lambda: {MITOSIS: [], IMPOSTER: []})
    for a in data["annotations"]:
        boxes[a["image_id"]][a["category_id"]].append(a["bbox"])  # x1,y1,x2,y2
    return images, boxes


def choose_split(images: dict, fig_ids: dict, per_type: int, test_frac: float,
                 val_frac: float, seed: int, holdout: list, base_per_type: int = 0):
    """Balanced sample per tumor type, split by image. Optional held-out tumor
    types go entirely to test (cross-domain evaluation).

    base_per_type: val/test sizes are computed as if only `base_per_type`
    images were sampled, and every extra image goes to train. Because the
    shuffle order does not depend on per_type, this reproduces the exact
    val/test images of an earlier `--per-type base_per_type` run, so results
    stay directly comparable while the training set grows."""
    rng = random.Random(seed)
    by_type = defaultdict(list)
    for im in images.values():
        if im["file_name"] in fig_ids:  # 50 images in the JSON have no download link
            by_type[im["tumor_type"]].append(im)

    rows = []
    for tumor in sorted(by_type):
        lst = sorted(by_type[tumor], key=lambda x: x["file_name"])
        rng.shuffle(lst)
        if per_type > 0:
            lst = lst[:per_type]
        if tumor in holdout:
            rows += [(im, "test") for im in lst]
            continue
        n = min(len(lst), base_per_type) if base_per_type else len(lst)
        n_test = max(1, round(n * test_frac))
        n_val = max(1, round(n * val_frac))
        for i, im in enumerate(lst):
            split = "test" if i < n_test else "val" if i < n_test + n_val else "train"
            rows.append((im, split))
    return rows


# --------------------------------------------------------------------------- #
# Download + tiling
# --------------------------------------------------------------------------- #
def download(fid: str, dest: Path, retries: int = 4):
    if dest.exists() and dest.stat().st_size > 0:
        return
    part = dest.with_suffix(dest.suffix + ".part")
    for attempt in range(1, retries + 1):
        try:
            urllib.request.urlretrieve(FIGSHARE_URL.format(fid), part)
            part.rename(dest)
            return
        except Exception as e:  # network hiccups are common on long runs
            print(f"  ! download {dest.name} failed ({e}), attempt {attempt}/{retries}")
            time.sleep(5 * attempt)
    raise RuntimeError(f"Could not download {dest.name}")


def read_rgb(path: Path) -> np.ndarray:
    try:
        import tifffile
        arr = tifffile.imread(str(path))
    except Exception:
        arr = np.array(Image.open(path))
    if arr.ndim == 2:
        arr = np.stack([arr] * 3, axis=-1)
    if arr.shape[0] in (3, 4) and arr.shape[-1] not in (3, 4):  # channel-first
        arr = np.moveaxis(arr, 0, -1)
    arr = arr[..., :3]
    if arr.dtype != np.uint8:
        arr = (arr / arr.max() * 255).astype(np.uint8)
    return arr


def crop_centered(arr, cx, cy, size):
    """size x size crop centred on (cx, cy); parts outside the image are white."""
    h, w = arr.shape[:2]
    x0, y0 = int(round(cx)) - size // 2, int(round(cy)) - size // 2
    out = np.full((size, size, 3), 255, np.uint8)
    sx0, sy0, sx1, sy1 = max(0, x0), max(0, y0), min(w, x0 + size), min(h, y0 + size)
    if sx1 > sx0 and sy1 > sy0:
        out[sy0 - y0:sy1 - y0, sx0 - x0:sx1 - x0] = arr[sy0:sy1, sx0:sx1]
    return out


def save_cell_crops(arr, im_boxes, stem, split, out, size):
    """Crops around every labelled mitosis (positive) and imposter (hard
    negative) for training the second-stage classifier."""
    n = Counter()
    for cat, label in ((MITOSIS, "mitosis"), (IMPOSTER, "imposter")):
        d = out / "crops" / split / label
        for j, (x1, y1, x2, y2) in enumerate(im_boxes[cat]):
            cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
            if not (0 <= cx < arr.shape[1] and 0 <= cy < arr.shape[0]):
                continue
            c = crop_centered(arr, cx, cy, size)
            Image.fromarray(c).save(d / f"{stem}_{j}.jpg", quality=95)
            n[label] += 1
    return n


def tile_starts(length: int, tile: int, stride: int):
    if length <= tile:
        return [0]
    starts = list(range(0, length - tile + 1, stride))
    if starts[-1] != length - tile:  # last tile flush with the border
        starts.append(length - tile)
    return starts


def boxes_in_tile(boxes, x0, y0, tile):
    """Boxes whose CENTER lies inside the tile, clipped to the tile."""
    out = []
    for x1, y1, x2, y2 in boxes:
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        if x0 <= cx < x0 + tile and y0 <= cy < y0 + tile:
            bx1, by1 = max(x1, x0) - x0, max(y1, y0) - y0
            bx2, by2 = min(x2, x0 + tile) - x0, min(y2, y0 + tile) - y0
            if bx2 - bx1 > 4 and by2 - by1 > 4:
                out.append((bx1, by1, bx2, by2))
    return out


def tile_image(arr, im_boxes, stem, split, out, tile, overlap, neg_ratio, rng):
    h, w = arr.shape[:2]
    stride = tile - overlap
    stats = Counter()
    img_dir, lbl_dir = out / "images" / split, out / "labels" / split
    for y0 in tile_starts(h, tile, stride):
        for x0 in tile_starts(w, tile, stride):
            mit = boxes_in_tile(im_boxes[MITOSIS], x0, y0, tile)
            imp = boxes_in_tile(im_boxes[IMPOSTER], x0, y0, tile)
            crop = arr[y0:y0 + tile, x0:x0 + tile]
            if crop.mean() > 225:  # mostly empty glass / background
                stats["skipped_background"] += 1
                continue
            keep = bool(mit) or bool(imp) or rng.random() < neg_ratio
            if not keep:
                stats["skipped_negative"] += 1
                continue
            if crop.shape[0] != tile or crop.shape[1] != tile:  # tiny images: pad
                pad = np.full((tile, tile, 3), 255, np.uint8)
                pad[:crop.shape[0], :crop.shape[1]] = crop
                crop = pad
            name = f"{stem}_x{x0}_y{y0}"
            Image.fromarray(crop).save(img_dir / f"{name}.jpg", quality=95)
            with open(lbl_dir / f"{name}.txt", "w") as f:
                for bx1, by1, bx2, by2 in mit:
                    cx, cy = (bx1 + bx2) / 2 / tile, (by1 + by2) / 2 / tile
                    bw, bh = (bx2 - bx1) / tile, (by2 - by1) / tile
                    f.write(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n")
            stats["tiles"] += 1
            stats["tiles_with_mitosis"] += bool(mit)
            stats["tiles_hard_negative"] += (not mit) and bool(imp)
            stats["mitoses"] += len(mit)
    return stats


# --------------------------------------------------------------------------- #
# Main
# --------------------------------------------------------------------------- #
def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--repo", type=Path, required=True, help="Path to the cloned MIDOGpp repository")
    ap.add_argument("--out", type=Path, default=Path("datasets/midog_yolo"))
    ap.add_argument("--per-type", type=int, default=10, help="Images per tumor type (0 = all)")
    ap.add_argument("--tile", type=int, default=640)
    ap.add_argument("--overlap", type=int, default=64)
    ap.add_argument("--neg-ratio", type=float, default=0.15, help="Fraction of empty tiles kept as negatives")
    ap.add_argument("--test-frac", type=float, default=0.2)
    ap.add_argument("--val-frac", type=float, default=0.1)
    ap.add_argument("--holdout", nargs="*", default=[], help="Tumor types sent entirely to test (cross-domain)")
    ap.add_argument("--workers", type=int, default=4, help="Parallel downloads")
    ap.add_argument("--tmp-dir", type=Path, default=None, help="Where TIFFs are downloaded before tiling")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--images-dir", type=Path, default=None,
                    help="Folder with already-downloaded TIFFs (skips downloading those)")
    ap.add_argument("--dry-run", action="store_true", help="Only print the planned split")
    ap.add_argument("--base-per-type", type=int, default=0,
                    help="Keep the val/test images of an earlier run with this --per-type; extra images go to train")
    ap.add_argument("--keep-split", type=Path, default=None,
                    help="split.csv of an earlier run: verify val/test are identical (stops if not)")
    ap.add_argument("--crop-size", type=int, default=0,
                    help="If > 0, also save crops of mitoses/imposters (train + val) for a second-stage classifier")
    args = ap.parse_args()

    fig_ids = load_figshare_ids(args.repo)
    images, boxes = load_annotations(args.repo)
    rows = choose_split(images, fig_ids, args.per_type, args.test_frac, args.val_frac, args.seed, args.holdout,
                        args.base_per_type)

    if args.keep_split:
        with open(args.keep_split) as f:
            old = {(r["file_name"], r["split"]) for r in csv.DictReader(f)}
        new = {(im["file_name"], s) for im, s in rows}
        old_eval = {x for x in old if x[1] != "train"}
        new_eval = {x for x in new if x[1] != "train"}
        old_train = {n for n, s in old if s == "train"}
        new_train = {im["file_name"] for im, s in rows if s == "train"}
        if old_eval != new_eval or not old_train <= new_train:
            sys.exit(f"Split mismatch with {args.keep_split}: val/test differ or old train images missing")
        print(f"Verified: same {len(new_eval)} val/test images as {args.keep_split}, "
              f"train grows {len(old_train)} -> {len(new_train)} images")

    plan = Counter((im["tumor_type"], s) for im, s in rows)
    print(f"Selected {len(rows)} images (~{len(rows) * 0.13:.0f} GB to download)")
    for tumor in sorted({t for t, _ in plan}):
        print(f"  {tumor:<35} train {plan[(tumor, 'train')]:>3}  val {plan[(tumor, 'val')]:>3}  test {plan[(tumor, 'test')]:>3}")
    if args.dry_run:
        return

    out = args.out.resolve()
    for split in ("train", "val", "test"):
        (out / "images" / split).mkdir(parents=True, exist_ok=True)
        (out / "labels" / split).mkdir(parents=True, exist_ok=True)
        (out / "full" / split).mkdir(parents=True, exist_ok=True)
    if args.crop_size:
        for split in ("train", "val"):
            for label in ("mitosis", "imposter"):
                (out / "crops" / split / label).mkdir(parents=True, exist_ok=True)
    tmp = args.tmp_dir or (out / "_download")
    tmp.mkdir(parents=True, exist_ok=True)

    with open(out / "split.csv", "w", newline="") as f:
        wr = csv.writer(f)
        wr.writerow(["file_name", "image_id", "tumor_type", "split", "width", "height", "n_mitoses"])
        for im, s in rows:
            wr.writerow([im["file_name"], im["id"], im["tumor_type"], s, im["width"], im["height"],
                         len(boxes[im["id"]][MITOSIS])])

    done_file = out / "_done.txt"
    done = set(done_file.read_text().split()) if done_file.exists() else set()
    lock = threading.Lock()
    totals = defaultdict(Counter)
    todo = [(im, s) for im, s in rows if im["file_name"] not in done]
    print(f"{len(done)} images already processed, {len(todo)} to go")

    def process(item):
        im, split = item
        name = im["file_name"]
        local = args.images_dir / name if args.images_dir and (args.images_dir / name).exists() else None
        src = local or (tmp / name)
        if local is None:
            download(fig_ids[name], src)
        arr = read_rgb(src)
        rng = random.Random(f"{args.seed}-{name}")
        st = tile_image(arr, boxes[im["id"]], Path(name).stem, split, out,
                        args.tile, args.overlap, args.neg_ratio, rng)
        if args.crop_size and split in ("train", "val"):
            st.update(save_cell_crops(arr, boxes[im["id"]], Path(name).stem, split, out, args.crop_size))
        del arr
        if split in ("val", "test"):  # keep full images for whole-image evaluation
            dest = out / "full" / split / name
            if local:
                shutil.copy2(src, dest)
            else:
                shutil.move(str(src), dest)
        elif local is None:
            src.unlink(missing_ok=True)
        with lock:
            with open(done_file, "a") as f:
                f.write(name + "\n")
        return name, split, st

    failed = []
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=args.workers) as pool:
        futures = [pool.submit(process, it) for it in todo]
        for k, fut in enumerate(as_completed(futures), 1):
            try:
                name, split, st = fut.result()
            except Exception as e:
                print(f"  ! {e} (re-run the script to retry)")
                failed.append(str(e))
                continue
            totals[split].update(st)
            el = time.time() - t0
            eta = el / k * (len(todo) - k) / 60
            print(f"[{k}/{len(todo)}] {name} ({split}): {st['tiles']} tiles, "
                  f"{st['mitoses']} mitoses | elapsed {el / 60:.1f} min, ETA {eta:.1f} min")

    shutil.rmtree(tmp, ignore_errors=True)

    yaml = (f"# MIDOG++ mitotic figure detection (YOLO format), built by prepare_midog.py\n"
            f"path: {out.as_posix()}\ntrain: images/train\nval: images/val\ntest: images/test\n\n"
            f"names:\n  0: mitotic figure\n")
    (out / "data.yaml").write_text(yaml)

    # statistics straight from disk, so they are correct after resumed runs too
    manifest = {"args": {k: str(v) for k, v in vars(args).items()}, "splits": {}}
    for split in ("train", "val", "test"):
        lbls = list((out / "labels" / split).glob("*.txt"))
        n_obj = [len(p.read_text().splitlines()) for p in lbls]
        manifest["splits"][split] = {
            "images": sum(1 for _, s in rows if s == split),
            "tiles": len(lbls),
            "tiles_with_mitosis": sum(1 for n in n_obj if n),
            "mitoses_in_tiles": sum(n_obj),
        }
        if args.crop_size and split != "test":
            for label in ("mitosis", "imposter"):
                manifest["splits"][split][f"crops_{label}"] = len(list((out / "crops" / split / label).glob("*.jpg")))
    (out / "manifest.json").write_text(json.dumps(manifest, indent=2))
    print("\nDone.", json.dumps(manifest["splits"], indent=2))
    print(f"Dataset config: {out / 'data.yaml'}")
    if failed:
        print(f"\nWARNING: {len(failed)} image(s) failed. Run the same command again to retry only those.")
    manifest["failed"] = failed
    (out / "manifest.json").write_text(json.dumps(manifest, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
# Find Session A's split.csv (if attached) to verify val/test are unchanged
import glob
old = [p for p in glob.glob('/kaggle/input/**/split.csv', recursive=True)]
KEEP = f'--keep-split {old[0]}' if old else ''
print('Verifying against:', old[0] if old else 'nothing attached (split is still deterministic)')
ARGS = f'--repo /tmp/MIDOGpp --out {OUT} --per-type {PER_TYPE} --base-per-type {BASE} --crop-size {CROP} {KEEP}'

# Preview the plan (no download yet)
!python /tmp/prepare_midog.py {ARGS} --dry-run

In [ ]:
# Download + tile + crops (the long step, roughly 15 to 30 minutes)
!python /tmp/prepare_midog.py {ARGS} --workers {WORKERS} --tmp-dir {TMP}

In [ ]:
# Automatic retry: only re-processes images that failed above (prints "0 to go" if everything worked)
!python /tmp/prepare_midog.py {ARGS} --workers 2 --tmp-dir {TMP}

In [ ]:
# Sanity check: draw labels on random tiles that contain mitoses -> label_preview.jpg
import glob, random, json, shutil
from IPython.display import display
from pathlib import Path
from PIL import Image, ImageDraw

random.seed(1)
pos = [p for p in glob.glob(f'{OUT}/labels/train/*.txt') if Path(p).read_text().strip()]
print(f'{len(pos)} training tiles contain mitoses')
grid = Image.new('RGB', (4 * 320, 2 * 320), 'white')
for i, lbl in enumerate(random.sample(pos, min(8, len(pos)))):
    im = Image.open(lbl.replace('/labels/', '/images/').replace('.txt', '.jpg')).convert('RGB')
    d = ImageDraw.Draw(im)
    for line in Path(lbl).read_text().splitlines():
        _, x, y, w, h = map(float, line.split())
        d.rectangle([(x - w / 2) * 640, (y - h / 2) * 640, (x + w / 2) * 640, (y + h / 2) * 640], outline=(0, 255, 0), width=3)
    grid.paste(im.resize((320, 320)), ((i % 4) * 320, (i // 4) * 320))
grid.save(f'{WORK}/label_preview.jpg')

# classifier crops: top row = mitoses, bottom row = imposters (look-alikes)
cg = Image.new('RGB', (8 * 128, 2 * 128), 'white')
for r, lab in enumerate(('mitosis', 'imposter')):
    fs = sorted(glob.glob(f'{OUT}/crops/train/{lab}/*.jpg'))
    print(lab, len(fs), 'train crops')
    for i, f in enumerate(random.sample(fs, min(8, len(fs)))):
        cg.paste(Image.open(f).convert('RGB'), (i * 128, r * 128))
cg.save(f'{WORK}/crop_preview.jpg')
display(grid); display(cg)

In [ ]:
# Package everything for Session B: one zip (Kaggle handles one big file better than thousands of small ones)
import os, json, shutil
shutil.copy('/tmp/MIDOGpp/databases/MIDOG++.json', f'{OUT}/MIDOG++.json')   # needed for full-image evaluation
for f in ('manifest.json', 'split.csv'):
    shutil.copy(f'{OUT}/{f}', f'{WORK}/{f}')
print(json.dumps(json.load(open(f'{OUT}/manifest.json'))['splits'], indent=2))
shutil.make_archive(f'{WORK}/midog_yolo_v2', 'zip', root_dir=WORK, base_dir='midog_yolo')
shutil.rmtree(OUT)
for f in sorted(os.listdir(WORK)):
    print(f'{f:<25} {os.path.getsize(os.path.join(WORK, f)) / 1e9:.2f} GB')
print('\nDone. Session B2 will use midog_yolo_v2.zip from this notebook\'s output.')